<a href="https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I use historical GSC and GA4 measurements as the feature vector for content refresh prioritization. I use impressions, clicks, average position, pageviews, and sessions. I also engineer CTR from clicks divided by impressions and pageviews per session from pageviews divided by sessions. Missing measurements are kept distinguishable from real zero values by retaining the data-availability flags before filling engineered numeric values. Client and content IDs are kept only as context and are not used as predictive features.

In [ ]:
import pandas as pd
import numpy as np

rel = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 is the working feature-development window
# established in ML-04.
df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_data_available,
    ga4_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

# Preserve availability information before filling values.
df["gsc_missing"] = (~df["gsc_data_available"].fillna(False)).astype(int)
df["ga4_missing"] = (~df["ga4_data_available"].fillna(False)).astype(int)

# Engineered features.
df["gsc_ctr"] = np.where(
    df["gsc_impressions"].fillna(0) > 0,
    df["gsc_clicks"].fillna(0) / df["gsc_impressions"].fillna(0),
    0.0
)

df["pageviews_per_session"] = np.where(
    df["ga4_sessions"].fillna(0) > 0,
    df["ga4_pageviews"].fillna(0) / df["ga4_sessions"].fillna(0),
    0.0
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "gsc_ctr",
    "pageviews_per_session",
    "gsc_missing",
    "ga4_missing",
]

X = df[feature_cols].copy()

# Numeric missing values are filled after availability flags are retained.
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Rows:", len(X))
print("Feature count:", len(feature_cols))
print("Features:", feature_cols)
print("Missing values remaining:", int(X.isna().sum().sum()))

display(X.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Feature count: 9
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'gsc_ctr', 'pageviews_per_session', 'gsc_missing', 'ga4_missing']
Missing values remaining: 0


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,gsc_ctr,pageviews_per_session,gsc_missing,ga4_missing
0,20,0,3.350000,0,0,0.000,0.0,0,1
1,1,0,0.000000,0,0,0.000,0.0,0,1
2,125,1,4.928000,0,0,0.008,0.0,0,1
3,7,0,4.000000,0,0,0.000,0.0,0,1
4,11,0,2.272727,0,0,0.000,0.0,0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

I use nine features in the vector. The five source measurements describe historical search and analytics performance. `gsc_ctr` and `pageviews_per_session` are engineered from those historical measurements. `gsc_missing` and `ga4_missing` preserve whether the corresponding data source was unavailable before numeric filling. All nine features are numeric after preprocessing. The feature values are from the March 2026 working window and are intended to be available before the prediction/refresh-prioritization decision. Client and content identifiers are not used as predictive features.

In [ ]:
# Feature notes backed by the actual feature vector.

feature_notes = pd.DataFrame([
    {
        "feature": "gsc_impressions",
        "meaning": "Search impressions for the content item",
        "missing_handling": "Filled with 0 after availability is retained",
        "available_before_prediction": True
    },
    {
        "feature": "gsc_clicks",
        "meaning": "Search clicks for the content item",
        "missing_handling": "Filled with 0 after availability is retained",
        "available_before_prediction": True
    },
    {
        "feature": "gsc_avg_position",
        "meaning": "Average search position",
        "missing_handling": "Filled with 0 when unavailable",
        "available_before_prediction": True
    },
    {
        "feature": "ga4_pageviews",
        "meaning": "Analytics pageviews",
        "missing_handling": "Filled with 0 after availability is retained",
        "available_before_prediction": True
    },
    {
        "feature": "ga4_sessions",
        "meaning": "Analytics sessions",
        "missing_handling": "Filled with 0 after availability is retained",
        "available_before_prediction": True
    },
    {
        "feature": "gsc_ctr",
        "meaning": "Clicks divided by impressions",
        "missing_handling": "Set to 0 when impressions are 0",
        "available_before_prediction": True
    },
    {
        "feature": "pageviews_per_session",
        "meaning": "Pageviews divided by sessions",
        "missing_handling": "Set to 0 when sessions are 0",
        "available_before_prediction": True
    },
    {
        "feature": "gsc_missing",
        "meaning": "Indicator that GSC data was unavailable",
        "missing_handling": "Boolean availability converted to 0/1",
        "available_before_prediction": True
    },
    {
        "feature": "ga4_missing",
        "meaning": "Indicator that GA4 data was unavailable",
        "missing_handling": "Boolean availability converted to 0/1",
        "available_before_prediction": True
    }
])

print("Feature count:", len(feature_notes))
print("All features numeric:", all(pd.api.types.is_numeric_dtype(X[c]) for c in feature_cols))
print("Missing values remaining:", int(X.isna().sum().sum()))

display(feature_notes)

Feature count: 9
All features numeric: True
Missing values remaining: 0


,feature,meaning,missing_handling,available_before_prediction
0,gsc_impressions,Search impressions for the content item,Filled with 0 after availability is retained,True
1,gsc_clicks,Search clicks for the content item,Filled with 0 after availability is retained,True
2,gsc_avg_position,Average search position,Filled with 0 when unavailable,True
3,ga4_pageviews,Analytics pageviews,Filled with 0 after availability is retained,True
4,ga4_sessions,Analytics sessions,Filled with 0 after availability is retained,True
5,gsc_ctr,Clicks divided by impressions,Set to 0 when impressions are 0,True
6,pageviews_per_session,Pageviews divided by sessions,Set to 0 when sessions are 0,True
7,gsc_missing,Indicator that GSC data was unavailable,Boolean availability converted to 0/1,True
8,ga4_missing,Indicator that GA4 data was unavailable,Boolean availability converted to 0/1,True


In [ ]:
# Check the observed range of the engineered ratio features.

print("Observed GSC CTR range:",
      round(X["gsc_ctr"].min(), 4),
      "to",
      round(X["gsc_ctr"].max(), 4))

print("Observed pageviews/session range:",
      round(X["pageviews_per_session"].min(), 4),
      "to",
      round(X["pageviews_per_session"].max(), 4))

print("GSC unavailable rows:", int(X["gsc_missing"].sum()))
print("GA4 unavailable rows:", int(X["ga4_missing"].sum()))

Observed GSC CTR range: 0.0 to 1.0
Observed pageviews/session range: 0.0 to 19.5
GSC unavailable rows: 6230317
GA4 unavailable rows: 9427412


## 3. The leakage hunt

I checked the feature vector for three main leakage risks: label-derived fields, future or overlapping information, and product or decision-derived fields. I also checked that client and content identifiers were not included as predictive features. The final feature vector contains only historical GSC/GA4 measurements and engineered transformations of those measurements. Based on these checks, I found no obvious label-derived, future-window, decision-derived, or identifier leakage in the feature vector.

In [ ]:
# Leakage hunt: check the final feature list against risky fields.

label_derived = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

decision_derived = {
    "refresh_score",
    "ranking_score",
    "priority_score",
    "recommended_action"
}

identifiers = {
    "client_id",
    "content_id",
    "client_hash_id",
    "content_hash_id"
}

used_features = set(feature_cols)

label_leakage = used_features.intersection(label_derived)
decision_leakage = used_features.intersection(decision_derived)
identifier_leakage = used_features.intersection(identifiers)

print("Label-derived fields found:", sorted(label_leakage))
print("Decision-derived fields found:", sorted(decision_leakage))
print("Identifier fields found:", sorted(identifier_leakage))

assert not label_leakage, "Label-derived leakage detected."
assert not decision_leakage, "Decision-derived leakage detected."
assert not identifier_leakage, "Identifier leakage detected."

print("\nLeakage field check: PASSED")

Label-derived fields found: []
Decision-derived fields found: []
Identifier fields found: []

Leakage field check: PASSED


In [ ]:
# Confirm that only documented features are used.

documented_features = set(feature_notes["feature"])

unexpected_features = used_features - documented_features
missing_documented_features = documented_features - used_features

print("Unexpected features:", sorted(unexpected_features))
print("Documented features missing from X:", sorted(missing_documented_features))

assert not unexpected_features
assert not missing_documented_features

print("\nFeature documentation consistency check: PASSED")

Unexpected features: []
Documented features missing from X: []

Feature documentation consistency check: PASSED


In [ ]:
# Check the feature-development time window.

feature_dates = pd.to_datetime(df["report_date"])

print("Earliest feature date:", feature_dates.min().date())
print("Latest feature date:", feature_dates.max().date())

assert feature_dates.min() >= pd.Timestamp("2026-03-01")
assert feature_dates.max() <= pd.Timestamp("2026-03-31")

print("\nMarch feature-window check: PASSED")

Earliest feature date: 2026-03-01
Latest feature date: 2026-03-31

March feature-window check: PASSED


## 4. What I excluded and why

I excluded fields that could leak the target, encode an existing decision, act only as identifiers, or represent information unavailable at prediction time.

- `trend_direction` — excluded because it is label-derived and would reveal the target.
- `trend_pct` — excluded because it is used to derive the decline label.
- `is_declining_label` — excluded because it is the target itself.
- `client_hash_id` — excluded as a predictive feature because it is an identifier; it is kept only for context or grouping.
- `content_hash_id` — excluded as a predictive feature because it identifies the content item rather than providing a generalizable signal.
- `report_date` — excluded from the feature vector because it is used for time-window control rather than as a predictive measurement.
- `client_has_gsc` — excluded because it is client-level availability metadata rather than a content-performance signal.
- `client_has_ga4` — excluded because it is client-level availability metadata rather than a content-performance signal.
- Future-period measurements — excluded because they would not be available at the prediction moment.
- Existing ranking, refresh, or priority scores — excluded because they would encode an existing decision rather than independent evidence.

In [ ]:
excluded_features = {
    "trend_direction": "label-derived; would reveal the target",
    "trend_pct": "used to derive the decline label",
    "is_declining_label": "target/label",
    "client_hash_id": "identifier; context/grouping only",
    "content_hash_id": "identifier; context/grouping only",
    "report_date": "time-window control, not a predictive measurement",
    "client_has_gsc": "client-level availability metadata",
    "client_has_ga4": "client-level availability metadata",
    "future-period measurements": "not available at prediction time",
    "existing ranking/refresh/priority scores": "decision-derived information",
}

print("Excluded fields and reasons:\n")

for field, reason in excluded_features.items():
    print(f"- {field}: {reason}")

print("\nTotal excluded categories:", len(excluded_features))

Excluded fields and reasons:

- trend_direction: label-derived; would reveal the target
- trend_pct: used to derive the decline label
- is_declining_label: target/label
- client_hash_id: identifier; context/grouping only
- content_hash_id: identifier; context/grouping only
- report_date: time-window control, not a predictive measurement
- client_has_gsc: client-level availability metadata
- client_has_ga4: client-level availability metadata
- future-period measurements: not available at prediction time
- existing ranking/refresh/priority scores: decision-derived information

Total excluded categories: 10


In [ ]:
# Final ML-05 feature-vector check

print("Final feature vector:")
print(feature_cols)

print("\nNumber of features:", len(feature_cols))
print("Rows:", len(X))
print("Missing values:", int(X.isna().sum().sum()))

assert len(feature_cols) == 9
assert X.isna().sum().sum() == 0

print("\nFINAL FEATURE VECTOR CHECK: PASSED")

Final feature vector:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'gsc_ctr', 'pageviews_per_session', 'gsc_missing', 'ga4_missing']

Number of features: 9
Rows: 9841378
Missing values: 0

FINAL FEATURE VECTOR CHECK: PASSED


## Self-check

- ☑️ Every section above is filled — markdown thinking AND the code that backs it.
- ☑️ The notebook runs top to bottom with no errors — verified with Runtime → Run all.
- ☑️ No client names, URLs, or private queries anywhere.
- ☑️ My claims use careful words: observed, measured, directional, decision-support.
- ☑️ Committed to my repo under `work/notebooks/`.
